# Prompting

Prompting in LLMs is the design of a structured input to provide task description, demostrations and the actual input for the model to generate a desired output.

In [1]:
%pip install datasets evaluate transformers accelerate peft bitsandbytes
%pip install sacrebleu
%pip install huggingface_hub
%pip install unbabel-comet

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In this notebook, we are going to use for prompting a dataset set that is already available in the [Datasets repository](https://huggingface.co/datasets) from Hugging Face. However, the [Datasets library](https://huggingface.co/docs/datasets) makes easy to access and load datasets. For example, you can easily load your own dataset following [this tutorial](https://huggingface.co/docs/datasets/loading#local-and-remote-files).

More precisely, we are going to explain how to perform In-Context Learning with the [Llama2 model](https://huggingface.co/docs/transformers/model_doc/llama2) on the [MultiUN dataset](https://huggingface.co/datasets/Helsinki-NLP/multiun), specifically the Russian to Chinese (ru-zh) subset.

In [2]:
from datasets import load_dataset, DatasetDict

# Load the MultiUN dataset for Russian-Chinese
raw_datasets = load_dataset("Helsinki-NLP/multiun", "ru-zh")

# MultiUN only has a train split, so we create validation and test splits
print(raw_datasets)

# We'll use: 10,000 for training, 1,000 for test, 1,000 for validation.
# First, select 12,000 samples from the dataset
raw_datasets["train"] = raw_datasets["train"].shuffle(seed=42).select(range(12000))

# Splitting: 10,000 for training, 2,000 remaining for validation and test
train_test = raw_datasets["train"].train_test_split(test_size=2000, seed=42)

# Splitting the 2,000 remaining: 1,000 for validation, 1,000 for test
test_val = train_test["test"].train_test_split(test_size=1000, seed=42)

# The dataset has a 'translation' column with 'ru' and 'zh' keys.
# We map it to 'source_text' and 'dest_text' to match the notebook structure.
def map_to_src_tgt(batch):
    return {
        "source_lang": ["ru"] * len(batch["translation"]),
        "dest_lang": ["zh"] * len(batch["translation"]),
        "source_text": [x["ru"] for x in batch["translation"]],
        "dest_text": [x["zh"] for x in batch["translation"]],
    }

raw_datasets = DatasetDict({
    "train": train_test["train"],
    "valid": test_val["train"],
    "test": test_val["test"]
})

raw_datasets = raw_datasets.map(map_to_src_tgt, batched=True, remove_columns=["translation"])

/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 9557007
    })
})


We have manually created the training, validation, and test splits from the original training set. Each set is a dictionary with a list of source sentences (source_text), target sentences (dest_text) and the target language (dest_lang).

Let's take a closer look at the features of the training set:

In [3]:
raw_datasets["train"].features

{'source_lang': Value('string'),
 'dest_lang': Value('string'),
 'source_text': Value('string'),
 'dest_text': Value('string')}

We are focusing on the translation from Russian to Chinese.

Let us take a look at the translations of the first two Russian sentences:

In [4]:
raw_datasets["train"][:14]["source_text"]

['Упрощения текста и подготовки прямых переводов Декларации на различные языки коренных народов будет явно недостаточно, и потребуется принять другие меры для создания потенциала в рамках общин коренных и некоренных народов.',
 'Достаточно назвать в этой связи Международный трибунал по бывшей Югославии, Международный уголовный трибунал по Руанде, Международный трибунал по морскому праву, Международный уголовный суд.',
 'Председатель: За проект резолюции подано 15\xa0голосов.',
 'Вот почему мы должны засучить рукава и решительно взяться за переделку этого органа, заседающего за столом в форме подковы.',
 'i) систематического совпадения проверок и пиков в накоплении углерода; и',
 'В библиотеках содержится почти 60\xa0млн.',
 '- придания эффективного характера трудовому законодательству и трудовым институтам, в том числе в отношении признания трудового правоотношения, содействия нормальным трудовым отношениям и создания эффективно действующих систем инспекции труда;',
 'Г-н\xa0Алкалай (Б

In [5]:
raw_datasets["train"][:14]["dest_text"]

['《宣言》可以提供必要的框架，用于召集地方、国家和区域三级的土著人民组织，集体取得人权成果。',
 '在这方面，我们只需提到前南斯拉夫问题国际刑事法庭、卢旺达问题国际刑事法庭、国际海洋法法庭和国际刑事法院。',
 '主席（以俄语发言）：有15票赞成。',
 '这就是为什么我们必须迅速拿起榔头和钉子，改造马蹄型会议桌。 二十一世纪不需要马蹄型会议桌，而是需要圆形桌，可以多放几把椅子。',
 '此后，应每隔五年进行核查和核证直至入计期结束。',
 '31所高等教育院校设有俄语和俄罗斯文学培训课程。',
 '- 使劳动法和机构富有成效，包括有关承认雇佣关系、促进良好的产业关系以及建立有效的劳动监察制度；和',
 '阿尔卡拉伊先生（波斯尼亚和黑塞哥维那）（以英语发言）：今天，我非常荣幸能与诸位一起在此开会，我要借此机会由衷地感谢联合国大会主席及菲律宾和巴基斯坦两国政府召开这次会议，讨论这一重要议题。',
 'WFP还参加UNSCN关于HIV/AIDS、家庭粮食安全、学校保健与营养、紧急情况中的营养和微量营养素等问题工作组的工作。',
 '15. 确认秘书长的斡旋在非洲起着重要作用，并鼓励秘书长继续尽可能经常运用调解手段来帮助和平解决冲突，并在这方面酌情与非洲联盟和其他次区域组织进行密切协作；',
 '2000年5月31日伊拉克代表给秘书长的信（S/2000/528）。',
 '全球环境基金理事会于2003年11月在华盛顿举行了会议，在会上请环境基金的首席执行官向理事会提交一份建议草案，以供审查和发表评论，提交的时间应足够提前，以便能够把理事会的意见反映在定于2005年提交第七届缔约国会议的谅解备忘录草稿之中。',
 '回历1424年3月27日-29日(2003年5月28日-30日)于伊朗伊斯兰共和国德黑兰举行的伊斯兰外交部长第三十届会议(团结与尊严会议)，',
 '四、信息和宣传']

In [6]:
raw_datasets["train"][:14]["dest_lang"]

['zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh',
 'zh']

We have prepared the dataset to contain Russian source texts and Chinese target texts.

The Llama2 model is a pretrained Large Language Model (LLM) ready to tackle several NLP tasks, being one of them the translation from Russian into Chinese. Since we have already selected the specific language pair (ru-zh), we don't need to perform additional filtering by language.

In [7]:
# Language codes are already set during dataset loading
lang="zh"

More precisely, we are going to be using the Llama-2 checkpoint [meta-llama/Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) to run our experiments for which you need to accept the LLAMA 2 COMMUNITY LICENSE AGREEMENT. Processing your request may take some time, so please do it in advance.

Logging in HuggingFace to be granted access to Llama2 with 7B parameters:

In [8]:
import os
from huggingface_hub import login

# Authenticate using token from environment variable
# To set the token, use: export HF_TOKEN="your_token_here"
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("✓ Authenticated with HuggingFace")
else:
    raise ValueError("HF_TOKEN environment variable not found. Please set it with: export HF_TOKEN='your_token'")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Authenticated with HuggingFace


We can apply the tokenizer function to any dataset taking advantage that Hugging Face Datasets are [Apache Arrow](https://arrow.apache.org) files stored on the disk, so you only keep the samples you ask for loaded in memory.

To keep the data as a dataset, we will use the [Dataset.map() function](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.map). This also allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset.

In our case, each sample pair is going to be preprocessed according to the needs of the model that is to be prompted. In the case of Llama2, it is recommended to explicitly state a task prompt for each source sentence:

In [9]:
from transformers import AutoTokenizer

max_tok_length = 512
checkpoint = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint,
    token=True,
    padding=True,
    pad_to_multiple_of=8,
    truncation=True,
    max_length=max_tok_length,
    padding_side='left',
    )
tokenizer.pad_token = tokenizer.eos_token

In [10]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["source_text"], 
        text_target = sample["dest_text"],
        )
    return model_inputs

The way the Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the tokenize function, that is, *input_ids*, *attention_mask* and *labels*. We can check what the preprocess_function is doing with a small sample

In [11]:
sample = raw_datasets["train"].select(range(2))
model_input = preprocess_function({
    "source_text": list(sample["source_text"]),
    "dest_text": list(sample["dest_text"]),
})
print(model_input)

{'input_ids': [[1, 2014, 5945, 14483, 23567, 1229, 606, 3693, 19300, 717, 23380, 5588, 29988, 2942, 984, 5752, 1453, 3506, 684, 494, 3540, 665, 16481, 20125, 14264, 717, 1046, 5719, 2430, 14528, 516, 23618, 29932, 2282, 29942, 570, 1538, 1802, 1229, 702, 4913, 29892, 606, 733, 11414, 3378, 4364, 17867, 1413, 28018, 757, 780, 29982, 3807, 14507, 1587, 733, 12087, 1138, 24036, 490, 1345, 29959, 9666, 24761, 29921, 1046, 5719, 2430, 606, 17379, 5719, 2430, 14528, 516, 29889], [1, 6546, 1229, 702, 4913, 10830, 1413, 490, 24643, 26874, 28064, 12133, 2370, 10357, 3378, 20222, 733, 2188, 29942, 14337, 6885, 588, 12329, 1221, 29917, 29892, 28064, 12133, 2370, 863, 588, 3176, 2370, 10357, 3378, 20222, 733, 8978, 745, 1216, 29892, 28064, 12133, 2370, 10357, 3378, 20222, 733, 15256, 18976, 16901, 29960, 29892, 28064, 12133, 2370, 863, 588, 3176, 2370, 3404, 29957, 29889]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [12]:
for sample in model_input['input_ids']:
    print(tokenizer.convert_ids_to_tokens(sample))

['<s>', '▁У', 'про', 'щения', '▁тек', 'ста', '▁и', '▁под', 'готов', 'ки', '▁пря', 'мы', 'х', '▁пере', 'во', 'дов', '▁Д', 'ек', 'ла', 'ра', 'ции', '▁на', '▁разли', 'чные', '▁язы', 'ки', '▁ко', 'рен', 'ных', '▁народ', 'ов', '▁буде', 'т', '▁я', 'в', 'но', '▁не', 'до', 'ста', 'то', 'чно', ',', '▁и', '▁по', 'тре', 'бу', 'ется', '▁приня', 'ть', '▁другие', '▁м', 'ер', 'ы', '▁для', '▁созда', 'ния', '▁по', 'тен', 'ци', 'ала', '▁в', '▁ра', 'м', 'ках', '▁общи', 'н', '▁ко', 'рен', 'ных', '▁и', '▁неко', 'рен', 'ных', '▁народ', 'ов', '.']
['<s>', '▁До', 'ста', 'то', 'чно', '▁назва', 'ть', '▁в', '▁этой', '▁связи', '▁Между', 'народ', 'ный', '▁три', 'бу', 'нал', '▁по', '▁бы', 'в', 'шей', '▁Ю', 'го', 'сла', 'ви', 'и', ',', '▁Между', 'народ', 'ный', '▁у', 'го', 'лов', 'ный', '▁три', 'бу', 'нал', '▁по', '▁Ру', 'ан', 'де', ',', '▁Между', 'народ', 'ный', '▁три', 'бу', 'нал', '▁по', '▁мор', 'скому', '▁прав', 'у', ',', '▁Между', 'народ', 'ный', '▁у', 'го', 'лов', 'ный', '▁су', 'д', '.']


We can recover the source text by applying [batch_decode](https://huggingface.co/docs/transformers/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.batch_decode) of the tokenizer 

In [13]:
tokenizer.batch_decode(model_input['input_ids'])

['<s> Упрощения текста и подготовки прямых переводов Декларации на различные языки коренных народов будет явно недостаточно, и потребуется принять другие меры для создания потенциала в рамках общин коренных и некоренных народов.',
 '<s> Достаточно назвать в этой связи Международный трибунал по бывшей Югославии, Международный уголовный трибунал по Руанде, Международный трибунал по морскому праву, Международный уголовный суд.']

Now, we can apply the preprocess_function to the raw datasets (training, validation and test):

In [14]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:  20%|██        | 2000/10000 [00:00<00:00, 18654.78 examples/s]

Map:  50%|█████     | 5000/10000 [00:00<00:00, 19638.61 examples/s]

Map:  70%|███████   | 7000/10000 [00:00<00:00, 19352.80 examples/s]

Map:  90%|█████████ | 9000/10000 [00:00<00:00, 18622.15 examples/s]

Map: 100%|██████████| 10000/10000 [00:00<00:00, 17969.55 examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map: 100%|██████████| 1000/1000 [00:00<00:00, 16025.80 examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map: 100%|██████████| 1000/1000 [00:00<00:00, 18436.83 examples/s]

We are going to filter the tokenized datasets by maximum number of tokens in source and target language:

In [15]:
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) <= max_tok_length and len(x["labels"]) <= max_tok_length , desc=f"Discarding source and target sentences with more than {max_tok_length} tokens")

Discarding source and target sentences with more than 512 tokens:   0%|          | 0/10000 [00:00<?, ? examples/s]

Discarding source and target sentences with more than 512 tokens:  40%|████      | 4000/10000 [00:00<00:00, 22001.43 examples/s]

Discarding source and target sentences with more than 512 tokens:  70%|███████   | 7000/10000 [00:00<00:00, 21851.80 examples/s]

Discarding source and target sentences with more than 512 tokens: 100%|██████████| 10000/10000 [00:00<00:00, 19823.70 examples/s]

Discarding source and target sentences with more than 512 tokens: 100%|██████████| 10000/10000 [00:00<00:00, 20309.09 examples/s]

Discarding source and target sentences with more than 512 tokens:   0%|          | 0/1000 [00:00<?, ? examples/s]

Discarding source and target sentences with more than 512 tokens: 100%|██████████| 1000/1000 [00:00<00:00, 20630.19 examples/s]

Discarding source and target sentences with more than 512 tokens:   0%|          | 0/1000 [00:00<?, ? examples/s]

Discarding source and target sentences with more than 512 tokens: 100%|██████████| 1000/1000 [00:00<00:00, 22430.63 examples/s]

We can take a quick look at the length histogram in the source language:

In [16]:
dic = {}
for sample in tokenized_datasets['train']:
    sample_length = len(sample['input_ids'])
    if sample_length not in dic:
        dic[sample_length] = 1
    else:
        dic[sample_length] += 1 

for i in range(1,max_tok_length+1):
    if i in dic:
        print(f"{i:>2} {dic[i]:>3}")

 2   8
 3  32
 4  93
 5  80
 6 117
 7 110
 8  93
 9  88
10 100
11 108
12 104
13 114
14 103
15 103
16 111
17 120
18 111
19  87
20  93
21  94
22 111
23 103
24  88
25 102
26 119
27 104
28 105
29 108
30 113
31 106
32 127
33 115
34 106
35 111
36 116
37 122
38 107
39 110
40 116
41 120
42 114
43 130
44 137
45 119
46 103
47 119
48  87
49 118
50  84
51 120
52  96
53 101
54 104
55 115
56 103
57  98
58  94
59  99
60  88
61  90
62 102
63  90
64  96
65 101
66  68
67  85
68  75
69 105
70  93
71  85
72  74
73  72
74  76
75  71
76  83
77  90
78  76
79  74
80  64
81  63
82  61
83  59
84  64
85  68
86  63
87  60
88  44
89  50
90  70
91  44
92  46
93  38
94  54
95  48
96  33
97  40
98  45
99  38
100  38
101  40
102  36
103  24
104  25
105  31
106  31
107  31
108  26
109  30
110  22
111  19
112  36
113  21
114  31
115  22
116  17
117  23
118  30
119  19
120  15
121  23
122  22
123   9
124  10
125  25
126  20
127  18
128  21
129   6
130  14
131  16
132  11
133   8
134  10
135  18
136  10
137   6
138   8
13

Checking a sample after filtering by maximum number of tokens:

In [17]:
for sample in tokenized_datasets['train'].select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[1, 2014, 5945, 14483, 23567, 1229, 606, 3693, 19300, 717, 23380, 5588, 29988, 2942, 984, 5752, 1453, 3506, 684, 494, 3540, 665, 16481, 20125, 14264, 717, 1046, 5719, 2430, 14528, 516, 23618, 29932, 2282, 29942, 570, 1538, 1802, 1229, 702, 4913, 29892, 606, 733, 11414, 3378, 4364, 17867, 1413, 28018, 757, 780, 29982, 3807, 14507, 1587, 733, 12087, 1138, 24036, 490, 1345, 29959, 9666, 24761, 29921, 1046, 5719, 2430, 606, 17379, 5719, 2430, 14528, 516, 29889]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 29871, 30866, 232, 177, 166, 31243, 30843, 30682, 30651, 31302, 231, 193, 158, 31641, 30698, 30210, 233, 164, 137, 233, 161, 185, 30214, 30406, 30909, 232, 146, 175, 30893, 30533, 30525, 30330, 30356, 30613, 30503, 30467, 232, 162, 162, 30457, 234, 189, 170, 30210, 31181, 235, 148, 154, 30313, 30855, 312

In [18]:
src = "ru"
tgt = lang
task_prefix = f"Translate from {src} to {tgt}:\n"
num_shots = 1
shots = ""
s = ""

prefix_tok_len = len(tokenizer.encode(f"{task_prefix}{shots}{src}: {s} = {tgt}: "))
shot_tok_len   = len(tokenizer.encode(f"{src}: {s} = {tgt}: {s}\n"))
max_tok_len = prefix_tok_len
max_tok_len += num_shots * (shot_tok_len + 2 * max_tok_length) 
max_tok_len += max_tok_length

random_seed = 13
sample = tokenized_datasets['train'].shuffle(seed=random_seed).select(range(num_shots))
for s in sample: shots += f"{src}: {s['source_text']} = {tgt}: {s['dest_text']}\n" 

def preprocess4test_function(sample):
    inputs = [f"{task_prefix}{shots}{src}: {s} = {tgt}: " for s in sample["source_text"]]
    model_inputs = tokenizer(
        inputs,
        max_length=max_tok_len, 
        truncation=True, 
        return_tensors="pt", 
        padding=True)
    return model_inputs

The way the Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the tokenize function, that is, *input_ids*, *attention_mask* and *labels*:

In [19]:
sample = tokenized_datasets['test'].select(range(5))
model_input = preprocess4test_function(sample)
print(model_input)
print(tokenizer.batch_decode(model_input['input_ids']))

{'input_ids': tensor([[    2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     1,  4103,  9632,
           515,  5796,   304,   503, 29882, 29901,    13,   582, 29901, 25131,
         10786,  6792, 10865,  6052,   665,  3982, 10046,  2815, 11397,  3325,
          9960,  9480, 18972,  6587,  8535, 29889,   353,   503, 29882, 29901,
         29871, 29946, 29955, 29889, 29871, 30602,   232,   181,   187, 30210,
           234,   141,   188, 30654, 3

In [20]:
preprocessed_test_dataset = tokenized_datasets['test'].map(preprocess4test_function, batched=True)

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Map: 100%|██████████| 999/999 [00:00<00:00, 6542.50 examples/s]

Map: 100%|██████████| 999/999 [00:00<00:00, 6372.34 examples/s]

In [21]:
for sample in preprocessed_test_dataset.select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 4103, 9632, 515, 5796, 304, 503, 29882, 29901, 13, 582, 29901, 25131, 10786, 6792, 10865, 6052, 665, 3982, 10046, 2815, 11397, 3325, 9960, 9480, 18972, 6587, 8535, 29889, 353, 503, 29882, 29901, 29871, 29946, 29955, 29889, 29871, 30602, 232, 181, 187, 30210, 234, 141, 188, 30654, 30313, 30495, 31924, 30940, 30392, 31838, 30545, 30210, 30267, 13, 582, 29901, 

bitsandbytes is a quantization library with a Transformers integration. With this integration, you can quantize a model to 8 or 4-bits and enable many other options by configuring the BitsAndBytesConfig class. For example, you can:

<ul>
<li>set load_in_4bit=True to quantize the model to 4-bits when you load it</li>
<li>set bnb_4bit_quant_type="nf4" to use a special 4-bit data type for weights initialized from a normal distribution</li>
<li>set bnb_4bit_use_double_quant=True to use a nested quantization scheme to quantize the already quantized weights</li>
<li>set bnb_4bit_compute_dtype=torch.bfloat16 to use bfloat16 for faster computation</li>
</ul>


In [22]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

Pass the quantization_config to the from_pretrained method.

In [23]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    token=True,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:07<00:07,  7.80s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:10<00:00,  4.78s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:10<00:00,  5.23s/it]

# Inference

Loading default inference parameters for the model, so that additional parameters could be added and passed to the [generate function](https://huggingface.co/docs/transformers/main_classes/text_generation):

In [24]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
    )

print(generation_config)

GenerationConfig {
  "bos_token_id": 1,
  "do_sample": true,
  "eos_token_id": 2,
  "max_length": 4096,
  "pad_token_id": 0,
  "temperature": 0.6,
  "top_p": 0.9
}



As observed, the default search strategy for Llama-2 is Top-p with probability 0.9 and temperature 0.6 ($0<T<1$ amplifies output probability differences and makes output more deterministic). [The search strategy can be selected](https://huggingface.co/docs/transformers/en/generation_strategies) at inference time. 

First, the test set is divided in small batches to reduce GPU memory comsumption:

In [25]:
test_batch_size = 32
batch_tokenized_test = preprocessed_test_dataset.batch(test_batch_size)

Batching examples:   0%|          | 0/999 [00:00<?, ? examples/s]

Batching examples:   3%|▎         | 32/999 [00:00<00:09, 98.97 examples/s]

Batching examples:  51%|█████▏    | 512/999 [00:00<00:00, 1501.33 examples/s]

Batching examples: 100%|██████████| 999/999 [00:00<00:00, 2471.90 examples/s]

Batching examples: 100%|██████████| 999/999 [00:00<00:00, 1851.24 examples/s]

In [26]:
number_of_batches = len(batch_tokenized_test["input_ids"])
output_sequences = []
for i in range(number_of_batches):
    with torch.no_grad():
        output_batch = model.generate(
            generation_config=generation_config, 
            input_ids=torch.tensor(batch_tokenized_test["input_ids"][i]).cuda(), 
            attention_mask=torch.tensor(batch_tokenized_test["attention_mask"][i]).cuda(), 
            max_length = max_tok_len, 
            num_beams=1, 
            do_sample=False,)
    output_sequences.extend(output_batch)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


## Evaluation

The output of the model is automatically evaluated compared to the reference translations. To this purpose, we use the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics. In our case, we use the [BLEU metric](https://huggingface.co/spaces/evaluate-metric/bleu), or to be more precise, [sacreBLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu).

In [27]:
from evaluate import load

metric = load("sacrebleu")

The example below performs a basic post-processing to decode the predictions and extract the translation:

In [28]:
import re

def compute_metrics(sample, output_sequences):
    inputs = [f"{task_prefix}{shots}{src}: {s} = {tgt}: " for s in sample["source_text"]]
    preds = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)
    print(inputs)
    print(preds)
    for i, (input,pred) in enumerate(zip(inputs,preds)):
      pred = re.search(r'^.*\n',pred.removeprefix(input).lstrip())
      if pred is not None:
        preds[i] = pred.group()[:-1]
      else:
        preds[i] = ""
    print(sample["source_text"])
    print(sample["dest_text"])
    print(preds)
    result = metric.compute(predictions=preds, references=sample["dest_text"])
    result = {"bleu": result["score"]}
    return result

In [29]:
# Calcular BLEU
bleu_result = compute_metrics(preprocessed_test_dataset, output_sequences)

# Cargar COMET
from comet import download_model, load_from_checkpoint
comet_model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_model_path)

# Preparar datos para COMET
import re
task_prefix = f"Translate from ru to zh:\n"
src = "ru"
tgt = "zh"

inputs = [f"{task_prefix}{shots}{src}: {s} = {tgt}: " for s in preprocessed_test_dataset["source_text"]]
preds = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)

# Post-processing para extraer solo la traducción
for i, (input, pred) in enumerate(zip(inputs, preds)):
    pred_match = re.search(r'^.*\n', pred.removeprefix(input).lstrip())
    if pred_match is not None:
        preds[i] = pred_match.group()[:-1]
    else:
        preds[i] = ""

# COMET requiere: source, hypothesis (predictions), reference
comet_input = []
for src_text, pred_text, ref_text in zip(preprocessed_test_dataset["source_text"], preds, preprocessed_test_dataset["dest_text"]):
    comet_input.append({
        "src": src_text,
        "mt": pred_text.strip(),
        "ref": ref_text.strip()
    })

# Calcular COMET
comet_result = comet_model.predict(comet_input, batch_size=8, gpus=1)

print(f'BLEU score: {bleu_result["bleu"]:.2f}')
print(f'COMET score: {comet_result["system_score"]:.4f}')

['Translate from ru to zh:\nru: Еврейские поселения на Западном берегу являются незаконными. = zh: 47. 西岸的犹太人定居点是非法的。\nru: Центральное место в КСУП ГМ занимает пропаганда национальных стратегий финансирования, оказавшихся успешными в других секторах. = zh: ', 'Translate from ru to zh:\nru: Еврейские поселения на Западном берегу являются незаконными. = zh: 47. 西岸的犹太人定居点是非法的。\nru: На женщинах война сказывается по-иному\xa0— Когда ВФВВ признала, что война сказывается на женщинах не так, как на мужчинах, она учредила в 1984\xa0году Комитет, который должен был выяснить, что несет и какие последствия имеет для женщин война и каким образом ВФВВ могла бы максимально эффективно решать связанные с ними проблемы. = zh: ', 'Translate from ru to zh:\nru: Еврейские поселения на Западном берегу являются незаконными. = zh: 47. 西岸的犹太人定居点是非法的。\nru: Затем в июне готовятся матрица БКР, а подготовка сметы расходов обычно осуществляется в период с июля по сентябрь данного года. = zh: ', 'Translate from ru t

/home/alumno.upv.es/scheng1/.conda/envs/RFA2526pt/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 77385.68it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


...

Predicting DataLoader 0:  98%|█████████▊| 123/125 [00:04<00:00, 26.68it/s]

Predicting DataLoader 0:  99%|█████████▉| 124/125 [00:04<00:00, 26.48it/s]

Predicting DataLoader 0: 100%|██████████| 125/125 [00:04<00:00, 26.30it/s]

Predicting DataLoader 0: 100%|██████████| 125/125 [00:04<00:00, 26.22it/s]

BLEU score: 9.97
COMET score: 0.6766
